# (a)Data import

In [1]:
import sqlite3
import numpy as np
import pandas as pd

# Connect to your SQLite database
db_path ="customer_churn.db"
conn = sqlite3.connect(db_path)

# Explicitly load your three original tables into clear pandas DataFrames
df_customer = pd.read_sql("SELECT * FROM db_customer", conn)
df_subscription = pd.read_sql("SELECT * FROM db_subscription", conn)
df_support = pd.read_sql("SELECT * FROM db_support", conn)

conn.close()

print("Data loaded successfully!")


Data loaded successfully!


# (b)Data Cleaning

## data cleaning on df_customer dataframe
## 1-rename column -name to customer_name 
## 2-drop columns interests and pincode 
## 3-change data type-column dob 
## 4-data strandardization-column gender 
## 5-fix missing values-column country 

In [2]:
# 1-rename column
df_customer.rename(columns={"name" : "customer_name"},inplace=True)
df_customer.head(5)

,customerid,customer_name,country,state,gender,dob,interests,pincode
0,0002-ORFBO,keshav,India,Maharashtra,Male,1982-04-12 00:00:00,travel,None
1,0003-MKNFE,raghav,India,Karnataka,Male,1995-11-23 00:00:00,NaN,None
2,0004-TLHLJ,lalita,India,Delhi,Female,1978-02-15 00:00:00,movie,None
3,0011-IGKFF,mohan,India,Nagaland,Male,2001-08-30 00:00:00,NaN,None
4,0013-EXCHZ,mira,India,Delhi,Female,1990-05-05 00:00:00,drama,None


In [3]:
# 2-drop columns interest and pincode
df_customer.drop(columns=["interests","pincode"],inplace=True)

In [4]:
#3-change data type-column dob
df_customer['dob'] = pd.to_datetime(df_customer['dob'])
df_customer.info()


<class 'pandas.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   customerid     21 non-null     str           
 1   customer_name  21 non-null     str           
 2   country        18 non-null     str           
 3   state          21 non-null     str           
 4   gender         21 non-null     str           
 5   dob            21 non-null     datetime64[us]
dtypes: datetime64[us](1), str(5)
memory usage: 1.1 KB


In [5]:
# 4-data strandardization-column gender
df_customer.gender.unique()
df_customer.gender=df_customer.gender.replace({"Male":"Men","Women":"Female"})
df_customer.gender


0        Men
1        Men
2     Female
3        Men
4     Female
5     Female
6     Female
7        Men
8     Female
9        Men
10       Men
11    Female
12    Female
13    Female
14       Men
15    Female
16    Female
17    Female
18       Men
19       Men
20    Female
Name: gender, dtype: str

In [6]:
# 5-fix missing values-column country
state_country_mapping = df_customer.dropna(subset=['country']).set_index('state')['country'].to_dict()
df_customer['country']=df_customer['country'].fillna(df_customer['state'].map(state_country_mapping))

In [7]:
df_customer.info()
print(f"df_customer is now cleaned and can be used for further analysis")

<class 'pandas.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   customerid     21 non-null     str           
 1   customer_name  21 non-null     str           
 2   country        21 non-null     str           
 3   state          21 non-null     str           
 4   gender         21 non-null     str           
 5   dob            21 non-null     datetime64[us]
dtypes: datetime64[us](1), str(5)
memory usage: 1.1 KB
df_customer is now cleaned and can be used for further analysis


### Save Cleaned Customer Data to Database 
### Push the fully cleaned `df_customer` DataFrame into SQLite as a permanent table named `customer_clean` for future SQL queries and analysis.

In [8]:
import sqlite3
conn = sqlite3.connect(db_path)
df_customer.to_sql("customer_clean", conn, if_exists="replace", index=False)
conn.close()
print("Saved successfully")

Saved successfully


In [9]:
import sqlite3
import pandas as pd

# 1. Connect to your database
conn = sqlite3.connect(db_path)

# 2. Select everything from your clean table
# This query tells SQL to fetch all the data from the table you just created
df_view = pd.read_sql("SELECT * FROM customer_clean", conn)

# 3. Close the connection
conn.close()

# 4. Display the result
df_view

,customerid,customer_name,country,state,gender,dob
0,0002-ORFBO,keshav,India,Maharashtra,Men,1982-04-12 00:00:00
1,0003-MKNFE,raghav,India,Karnataka,Men,1995-11-23 00:00:00
2,0004-TLHLJ,lalita,India,Delhi,Female,1978-02-15 00:00:00
3,0011-IGKFF,mohan,India,Nagaland,Men,2001-08-30 00:00:00
4,0013-EXCHZ,mira,India,Delhi,Female,1990-05-05 00:00:00
5,0013-MHZWF,durga,India,Delhi,Female,1988-12-10 00:00:00
6,0013-SMEOE,mina,India,Meghalaya,Female,1976-09-21 00:00:00
7,0014-BMAQU,madan,India,Rajasthan,Men,1999-03-14 00:00:00
8,0015-UOCOJ,maya,Nepal,Kathmandu,Female,1985-07-07 00:00:00
9,0016-QLJIS,arjun,Nepal,Kathmandu,Men,1993-10-29 00:00:00


In [10]:
df_subscription.info()

<class 'pandas.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 11 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   customerid               21 non-null     str    
 1   subscription_start_date  21 non-null     str    
 2   subscription_type        21 non-null     str    
 3   renewal_date             21 non-null     str    
 4   plan_type                21 non-null     str    
 5   contract_type            21 non-null     str    
 6   cancellation_date        6 non-null      str    
 7   cancellation_reason      6 non-null      str    
 8   monthly_charges          21 non-null     float64
 9   cltv                     21 non-null     int64  
 10  churn_score              21 non-null     int64  
dtypes: float64(1), int64(2), str(8)
memory usage: 1.9 KB


## data cleaning in df_subscription dataframe
## a-change data type of subscription_start_date,renewal_date and cancellation_date from str to datetime

In [11]:
df_subscription["subscription_start_date"]=pd.to_datetime(df_subscription["subscription_start_date"])
df_subscription["renewal_date"]=pd.to_datetime(df_subscription["renewal_date"])
df_subscription["cancellation_date"]=pd.to_datetime(df_subscription["cancellation_date"])
df_subscription.info()

<class 'pandas.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 11 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   customerid               21 non-null     str           
 1   subscription_start_date  21 non-null     datetime64[us]
 2   subscription_type        21 non-null     str           
 3   renewal_date             21 non-null     datetime64[us]
 4   plan_type                21 non-null     str           
 5   contract_type            21 non-null     str           
 6   cancellation_date        6 non-null      datetime64[us]
 7   cancellation_reason      6 non-null      str           
 8   monthly_charges          21 non-null     float64       
 9   cltv                     21 non-null     int64         
 10  churn_score              21 non-null     int64         
dtypes: datetime64[us](3), float64(1), int64(2), str(5)
memory usage: 1.9 KB


### Save Cleaned Subscription Data to Database
### Push the fully cleaned df_customer DataFrame into SQLite as a permanent table named subscription_clean for future SQL queries and analysis.

In [12]:
import sqlite3
conn = sqlite3.connect(db_path)
df_subscription.to_sql(
    "subscription_clean", conn, if_exists="replace", index=False
)
conn.close()
print("subscription_clean saved successfully!")

subscription_clean saved successfully!


In [13]:
df_support.info()

<class 'pandas.DataFrame'>
RangeIndex: 9 entries, 0 to 8
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   customerid      9 non-null      str   
 1   complaint_date  9 non-null      str   
 2   escalations     9 non-null      str   
 3   csat_score      9 non-null      int64 
 4   col_1           0 non-null      object
 5   comment         4 non-null      str   
dtypes: int64(1), object(1), str(4)
memory usage: 564.0+ bytes


## Data cleaning in df_support dataframe
## 1-change datatype of complaint_date from str to datetime
## 2-drop columns col_1,comment

In [14]:
# 1-change datatype of complaint_date from str to datetime¶
df_support["complaint_date"]=pd.to_datetime(df_support["complaint_date"])
df_support.info()


<class 'pandas.DataFrame'>
RangeIndex: 9 entries, 0 to 8
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   customerid      9 non-null      str           
 1   complaint_date  9 non-null      datetime64[us]
 2   escalations     9 non-null      str           
 3   csat_score      9 non-null      int64         
 4   col_1           0 non-null      object        
 5   comment         4 non-null      str           
dtypes: datetime64[us](1), int64(1), object(1), str(3)
memory usage: 564.0+ bytes


In [15]:
#2-drop columns col_1,comment¶
df_support.drop(columns=["col_1","comment"],inplace=True)
df_support.info()

<class 'pandas.DataFrame'>
RangeIndex: 9 entries, 0 to 8
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   customerid      9 non-null      str           
 1   complaint_date  9 non-null      datetime64[us]
 2   escalations     9 non-null      str           
 3   csat_score      9 non-null      int64         
dtypes: datetime64[us](1), int64(1), str(2)
memory usage: 420.0 bytes


### Save Cleaned Support Data to Database
### Push the fully cleaned df_support DataFrame into SQLite as a permanent table named support_clean for future SQL queries and analysis.

In [16]:
import sqlite3
import pandas as pd
conn=sqlite3.connect(db_path)
df_support.to_sql("support_clean",conn,if_exists="replace",index=False)
conn.close()
print("Saved successfully")


Saved successfully


In [17]:
import sqlite3
import pandas as pd

# Connect to database
conn = sqlite3.connect(db_path)

# Fetch all table names
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table';", conn)
conn.close()

# Display the tables
print("Tables currently in the database:")
print(tables)

Tables currently in the database:
                            name
0                    db_customer
1                db_subscription
2                     db_support
3  subscription_churn_flag_table
4           unique_support_table
5              master_file_table
6                 customer_clean
7             subscription_clean
8                  support_clean


## (C) Feature Engineering 

### 1-create a churn flag for subscription_clean table
### 2-as customerid is not unique in support_clean table perform data modeling
### 3-make one master table of all three cleaned table for later analysis

In [18]:
# 1-create a churn flag for subscription_clean table
import pandas as pd
import sqlite3 
conn = sqlite3.connect(db_path)
query1 = """
    select *, 
        case 
            when cancellation_date is not null then 1
            else 0 
        end as churn_flag
    from subscription_clean
"""
df_subscription_clean_churnflag = pd.read_sql(query1, conn)
df_subscription_clean_churnflag.head()

,customerid,subscription_start_date,subscription_type,renewal_date,plan_type,contract_type,cancellation_date,cancellation_reason,monthly_charges,cltv,churn_score,churn_flag
0,0002-ORFBO,2021-03-15 00:00:00,Refferal,2025-03-15 00:00:00,Standard,Annual,NaN,NaN,13.99,627,12,0
1,0003-MKNFE,2020-08-01 00:00:00,Paid,2024-08-01 00:00:00,Premium,Annual,2024-09-10 00:00:00,Switched to competitor,12.99,1150,91,1
2,0004-TLHLJ,2022-11-20 00:00:00,Organic,2025-11-20 00:00:00,Basic,Monthly,NaN,NaN,6.99,210,34,0
3,0011-IGKFF,2019-05-10 00:00:00,Paid,2025-05-10 00:00:00,Premium,Annual,NaN,NaN,22.99,1725,8,0
4,0013-EXCHZ,2023-01-05 00:00:00,Refferal,2024-01-05 00:00:00,Standard,Monthly,2024-02-28 00:00:00,Too expensive,13.99,195,88,1


In [19]:
# (2)as you can see customerid for support_clean is not unique so have to perform data modeling on this table
import sqlite3
import pandas as pd
conn = sqlite3.connect(db_path)
query1="""select * from support_clean"""
df_not_unique=pd.read_sql(query1,conn)
conn.close()
df_not_unique

,customerid,complaint_date,escalations,csat_score
0,0003-MKNFE,2024-08-28 00:00:00,N,60
1,0003-MKNFE,2024-08-28 00:00:00,Y,10
2,0013-EXCHZ,2024-01-20 00:00:00,Y,20
3,0013-MHZWF,2025-03-18 00:00:00,N,90
4,0013-SMEOE,2024-11-01 00:00:00,N,30
5,0017-IUDMW,2024-04-10 00:00:00,Y,25
6,0019-EFAEP,2024-09-27 00:00:00,Y,30
7,0022-TCJCI,2024-09-13 00:00:00,Y,10
8,0022-TCJCI,2024-09-14 00:00:00,N,90


In [20]:
# removing duplicates from customerid
import pandas as pd
import sqlite3

conn = sqlite3.connect(db_path)
query="""
WITH my_cte AS (
    SELECT 
        customerid,
        complaint_date,
        escalations,
        csat_score,
        COUNT(customerid) OVER (PARTITION BY customerid) AS complaint_count,
        ROW_NUMBER() OVER (
            PARTITION BY customerid 
            ORDER BY complaint_date DESC, escalations DESC
        ) AS rn
    FROM support_clean
)
SELECT 
    customerid,
    complaint_date,
    escalations,
    csat_score,
    complaint_count
FROM my_cte 
WHERE rn = 1
"""

df_support_unique = pd.read_sql(query, conn)
conn.close()

df_support_unique

,customerid,complaint_date,escalations,csat_score,complaint_count
0,0003-MKNFE,2024-08-28 00:00:00,Y,10,2
1,0013-EXCHZ,2024-01-20 00:00:00,Y,20,1
2,0013-MHZWF,2025-03-18 00:00:00,N,90,1
3,0013-SMEOE,2024-11-01 00:00:00,N,30,1
4,0017-IUDMW,2024-04-10 00:00:00,Y,25,1
5,0019-EFAEP,2024-09-27 00:00:00,Y,30,1
6,0022-TCJCI,2024-09-14 00:00:00,N,90,2


In [21]:
# (3) as subscription_clean is the one having churn_score,so performing left join on it with rest two table for one master_table
import sqlite3
import pandas as pd
conn = sqlite3.connect(db_path)
df_subscription_clean_churnflag.to_sql("subscription_churn_flag_table",conn,if_exists="replace",index=False)
df_support_unique.to_sql("unique_support_table",conn,if_exists="replace",index=False)
conn.close()


In [22]:
##one master table of all three tables
import pandas as pd
import sqlite3

conn = sqlite3.connect(db_path)

query = """
    SELECT 
        s.*,
        c.customer_name, c.country, c.state, c.gender, c.dob,
        sup.complaint_date, sup.escalations, sup.csat_score, sup.complaint_count
    FROM subscription_churn_flag_table AS s
    LEFT JOIN customer_clean AS c 
        ON s.customerid = c.customerid
    LEFT JOIN unique_support_table AS sup 
        ON s.customerid = sup.customerid
"""

df_master = pd.read_sql(query, conn)
conn.close()



# (D) Data Analysis

In [23]:
#final exported csv master table
df_master.to_csv("final_master_table.csv",index=False)

In [24]:
# making table of df_master df for data analysis
import pandas as pd
import sqlite3
conn=sqlite3.connect(db_path)
df_master.to_sql("master_file_table",conn,if_exists="replace",index=False)
conn.close()


### 1.Churn rate

In [25]:
import pandas as pd
import sqlite3
conn=sqlite3.connect(db_path)
query="""
select avg(churn_flag)*100 as churn_rate from master_file_table """
churn_rate=pd.read_sql(query,conn)
conn.close()
churn_rate


,churn_rate
0,28.571429


### 2.Retention rate

In [26]:
import pandas as pd
import sqlite3
conn=sqlite3.connect(db_path)
query=""" select (100-(avg(churn_flag)*100)) as retention_rate from master_file_table"""
retention_rate=pd.read_sql(query,conn)
conn.close()
retention_rate

,retention_rate
0,71.428571


### 3.Churn by plan_type


In [27]:
import pandas as pd
import sqlite3
conn=sqlite3.connect(db_path)
query=""" select plan_type,avg(churn_flag)*100 as churn_rate from master_file_table
group by plan_type
order by churn_rate desc"""
churn_by_plan_type=pd.read_sql(query,conn)
conn.close()
churn_by_plan_type


,plan_type,churn_rate
0,Basic,60.000000
1,Standard,22.222222
2,Premium,14.285714


### 4(a)Churn by state + sum(revenue) and count of users

In [29]:
import pandas as pd
import sqlite3
conn=sqlite3.connect(db_path)
query="""
select state,avg(churn_flag)*100 as churn_percent,count(customerid) as total_users,sum(monthly_charges) as total_revenue from master_file_table 
group by state
"""
df_by_state=pd.read_sql(query,conn)
conn.close()
df_by_state

,state,churn_percent,total_users,total_revenue
0,Delhi,25.000000,4,52.96
1,Karnataka,100.000000,2,20.98
2,Kathmandu,0.000000,2,20.98
3,Maharashtra,0.000000,3,50.97
4,Meghalaya,66.666667,3,42.97
5,Nagaland,0.000000,1,22.99
6,Rajasthan,0.000000,2,36.98
7,Telangana,50.000000,2,30.98
8,Uttar Pradesh,0.000000,2,115.98


### 4(b)Churn by subscription_type + sum(revenue) and count of users

In [30]:
import pandas as pd
import sqlite3
conn=sqlite3.connect(db_path)
query="""
select subscription_type,avg(churn_flag)*100 as churn_percent,count(customerid) as total_users,sum(monthly_charges) as total_revenue from master_file_table 
group by subscription_type
"""
df_by_subscription=pd.read_sql(query,conn)
conn.close()
df_by_subscription

,subscription_type,churn_percent,total_users,total_revenue
0,Organic,0.000000,9,145.91
1,Paid,16.666667,6,174.94
2,Refferal,83.333333,6,74.94


### 5.ARPU(Average revenue per user)

In [31]:
import pandas as pd
import sqlite3
conn=sqlite3.connect(db_path)
query="""select avg(monthly_charges) as ARPU from master_file_table"""
Arpu=pd.read_sql(query,conn)
conn.close()
Arpu

,ARPU
0,18.847143


### 6.Average tenure days

In [32]:
import pandas as pd
import sqlite3
conn=sqlite3.connect(db_path)
query = """
    SELECT 
        customerid,
        subscription_start_date,
        cancellation_date,
        CASE 
            WHEN cancellation_date IS NULL OR cancellation_date = '' THEN 
                -- Active user: calculate days from start date to current date
               Round(julianday('now') - julianday(subscription_start_date),0)
            ELSE 
                -- Churned user: calculate days from start date to cancellation date
                Round(julianday(cancellation_date) - julianday(subscription_start_date),0)
        END AS tenure_days
    FROM master_file_table
"""
df_tenure = pd.read_sql(query, conn)
conn.close()
print(f"Average tenure days : {df_tenure['tenure_days'].mean()}")

Average tenure days : 1528.7142857142858


### 7.Revenue at loss: revenue lost from churned users

In [33]:
import sqlite3
conn=sqlite3.connect(db_path)
query = """
    SELECT 
        SUM(monthly_charges) AS total_revenue,
        SUM(CASE WHEN churn_flag = 1 THEN monthly_charges ELSE 0 END) AS churned_revenue,
        SUM(CASE WHEN churn_flag = 0 THEN monthly_charges ELSE 0 END) AS retained_revenue
    FROM master_file_table
"""
revenue_breakdown = pd.read_sql(query, conn)
conn.close()
revenue_breakdown

,total_revenue,churned_revenue,retained_revenue
0,395.79,73.94,321.85


### 8.Escalation rate

In [34]:
import sqlite3
conn=sqlite3.connect(r"C:\Users\Aryan\Desktop\Data Analytics Python Project by Rishabh Mishra\customer_churn.db")
query = """
    SELECT 
        AVG(CASE WHEN escalations = 'Y' THEN 1.0 ELSE 0.0 END) * 100 AS escalation_percentage
    FROM master_file_table
"""
escalation_rate = pd.read_sql(query, conn)
conn.close()
escalation_rate


,escalation_percentage
0,19.047619


### 9.Avg complaint per user

In [35]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(db_path)
query = """
    SELECT 
        SUM(complaint_count) * 1.0 / COUNT(customerid) AS avg_complaints
    FROM master_file_table
"""
df = pd.read_sql(query, conn)
conn.close()
df

,avg_complaints
0,0.428571


### 10.Correlation between escalation and churn

In [36]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(db_path)
query = """
    SELECT 
        CASE WHEN escalations = 'Y' THEN 1 ELSE 0 END AS escalations_num,
        churn_flag
    FROM master_file_table
"""
df = pd.read_sql(query, conn)
conn.close()

correlation = df['escalations_num'].corr(df['churn_flag'])
print("Correlation between escalation vs churn is = ", round(correlation, 2))

Correlation between escalation vs churn is =  0.77


### 11.Creating a column using churn_score

In [37]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(db_path)
query = """
select *,case 
when churn_score<50 then "low"
when churn_score >=50 and churn_score<70 then "mid"
when churn_score>=70 then "high"
end as churn_risk
from master_file_table
"""
churn_risk_df=pd.read_sql(query,conn)
conn.close()
churn_risk_df[["churn_score","churn_risk"]].head()

,churn_score,churn_risk
0,12,low
1,91,high
2,34,low
3,8,low
4,88,high
